# Normalize Data
Detects lined vs blank pages from PDF, crops lines to **Final Data**, then normalizes all output to **FinalDataNormalized**.

## Imports

In [15]:
import warnings
warnings.filterwarnings('ignore')
import cv2
import numpy as np
import os
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import fitz
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d

## Paths

In [16]:
INPUT_PDF_DIR        = "whole image"
FINAL_DATA_DIR       = "Final Data"
FINAL_DATA_NORM_DIR  = "FinalDataNormalized"

SKIP_TOP     = 10   # px to skip after printed line
PAD_BOTTOM   = 16   # px to add below next printed line
COLUMN_TOKEN = "_COLUMNS"

os.makedirs(FINAL_DATA_DIR,     exist_ok=True)
os.makedirs(FINAL_DATA_NORM_DIR, exist_ok=True)

## PDF Loading & Page Detection

In [17]:
def pdf_to_bgr_pages(path, dpi=300):
    doc   = fitz.open(path)
    scale = dpi / 72
    mat   = fitz.Matrix(scale, scale)
    pages = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR)
        pages.append(img)
    return pages


def horizontal_line_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    h, w = binary.shape
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(1, w // 5), 1))
    lines_only = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return cv2.countNonZero(lines_only)


def select_pages(path, dpi=300):
    """Returns (lined_page, blank_page). blank_page is None if only 1 page."""
    pages = pdf_to_bgr_pages(path, dpi=dpi)
    if len(pages) == 1:
        print("  Single-page PDF — treating as lined, no blank page")
        return pages[0], None
    scores = [horizontal_line_score(p) for p in pages]
    lined_idx = int(np.argmax(scores))
    blank_idx = 1 - lined_idx
    print(f"  Line scores: {scores}  ->  lined=page {lined_idx+1}, blank=page {blank_idx+1}")
    return pages[lined_idx], pages[blank_idx]

## Crop Lines (Lined Page)

In [18]:
def detect_printed_lines(img, debug=False):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    row_sums = np.zeros(h, dtype=float)
    for kw in [w // 12, w // 8, w // 5]:
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, 1))
        mask = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
        row_sums += mask.sum(axis=1).astype(float)

    row_sums = uniform_filter1d(row_sums, size=5)

    min_dist  = h // 50
    threshold = row_sums.max() * 0.08
    peaks, _  = find_peaks(row_sums, height=threshold, distance=min_dist)

    if debug:
        fig, axes = plt.subplots(1, 2, figsize=(16, 10))
        vis = img.copy()
        for y in peaks:
            cv2.line(vis, (0, int(y)), (w, int(y)), (0, 0, 255), 2)
        axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Detected {len(peaks)} lines (red)")
        axes[0].axis("off")
        axes[1].plot(row_sums)
        axes[1].axhline(threshold, color="r", linestyle="--", label="threshold")
        for y in peaks:
            axes[1].axvline(y, color="g", alpha=0.5)
        axes[1].set_title("Row sum profile (green = detected lines)")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

    return sorted(peaks.tolist())


def detect_content_x_bounds(img):
    """Detect the left and right vertical frame lines and return (x_left, x_right)
    inner bounds so crops exclude the frame borders."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Detect vertical lines: kernel tall enough to span a large fraction of the page
    kh = max(1, h // 4)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kh))
    vert_only = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    col_sums = vert_only.sum(axis=0).astype(float)
    col_sums = uniform_filter1d(col_sums, size=5)

    if col_sums.max() == 0:
        return 0, w

    threshold = col_sums.max() * 0.3
    peaks, _ = find_peaks(col_sums, height=threshold, distance=w // 20)

    if len(peaks) < 2:
        return 0, w

    x_left  = int(peaks[0])  + 4
    x_right = int(peaks[-1]) - 4
    print(f"  Frame bounds detected: x_left={x_left}, x_right={x_right}")
    return x_left, x_right


def crop_lines(img, line_ys, out_dir, basename,
               skip_top=SKIP_TOP, pad_bottom=PAD_BOTTOM, x_bounds=None):
    h, w = img.shape[:2]
    xl = x_bounds[0] if x_bounds else 0
    xr = x_bounds[1] if x_bounds else w
    saved = []
    for i in range(len(line_ys) - 1):
        y1 = min(h, line_ys[i]     + skip_top)
        y2 = min(h, line_ys[i + 1] + pad_bottom)
        crop = img[y1:y2, xl:xr]
        name = f"{basename}_line_{i+1:02d}.png"
        path = os.path.join(out_dir, name)
        cv2.imwrite(path, crop)
        saved.append(path)
    return saved

## Column Detection (Blank Page)

In [19]:
def column_detection(img, out_dir, basename):
    """Detect columns, draw red rectangles on each, save with COLUMN_TOKEN."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Remove blobs touching image border (corner markers)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    clean = binary.copy()
    for lbl in range(1, n):
        x  = stats[lbl, cv2.CC_STAT_LEFT]
        y  = stats[lbl, cv2.CC_STAT_TOP]
        bw = stats[lbl, cv2.CC_STAT_WIDTH]
        bh = stats[lbl, cv2.CC_STAT_HEIGHT]
        if x == 0 or y == 0 or x + bw >= w or y + bh >= h:
            clean[labels == lbl] = 0

    col_sums = clean.sum(axis=0).astype(float)
    col_sums = uniform_filter1d(col_sums, size=w // 40)

    threshold = col_sums.max() * 0.05
    in_col = col_sums > threshold

    bands, start = [], None
    for x, val in enumerate(in_col):
        if val and start is None:
            start = x
        elif not val and start is not None:
            bands.append([start, x])
            start = None
    if start is not None:
        bands.append([start, w])

    merged, min_gap = [], w // 50
    for band in bands:
        if merged and band[0] - merged[-1][1] < min_gap:
            merged[-1][1] = band[1]
        else:
            merged.append(band)

    boxes = []
    for x1, x2 in sorted(merged, key=lambda b: b[0]):
        strip = clean[:, x1:x2]
        ys = np.where(strip.sum(axis=1) > 0)[0]
        if len(ys) == 0:
            continue
        boxes.append((x1, int(ys[0]), x2, int(ys[-1])))

    vis = img.copy()
    for idx, (x1, y1, x2, y2) in enumerate(boxes):
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 3)
        cv2.putText(vis, f"col {idx+1}", (x1 + 4, y1 + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

    out_name = f"{basename}{COLUMN_TOKEN}.png"
    out_path = os.path.join(out_dir, out_name)
    cv2.imwrite(out_path, vis)
    print(f"  Detected {len(boxes)} column(s) -> saved {out_name}")
    return boxes

## Image Normalization

In [20]:
def binarize_image(img: np.ndarray, method: str = "adaptive") -> np.ndarray:
    if method == "otsu":
        _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif method == "adaptive":
        binary = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 10)
    elif method == "sauvola":
        binary = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 25, 15)
    else:
        raise ValueError(f"Unknown binarization method: {method}")
    return binary


def remove_noise(binary_img: np.ndarray, min_component_size: int = 10) -> np.ndarray:
    kernel = np.ones((2, 2), np.uint8)
    opened = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel, iterations=1)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(255 - opened, connectivity=8)
    output = np.zeros_like(opened)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_component_size:
            output[labels == i] = 255
    return 255 - output


def crop_to_content(img: np.ndarray, padding: int = 20) -> np.ndarray:
    coords = cv2.findNonZero(255 - img)
    if coords is None:
        return img
    x, y, w, h = cv2.boundingRect(coords)
    x = max(0, x - padding)
    y = max(0, y - padding)
    w = min(img.shape[1] - x, w + 2 * padding)
    h = min(img.shape[0] - y, h + 2 * padding)
    return img[y:y+h, x:x+w]


def normalize_height(img: np.ndarray, target_height: int = 64) -> np.ndarray:
    h, w = img.shape[:2]
    new_width = int(target_height * (w / h))
    return cv2.resize(img, (new_width, target_height), interpolation=cv2.INTER_AREA)


def normalize_image(img_bgr: np.ndarray,
                    binarize: bool = True, denoise: bool = True,
                    crop: bool = True, normalize_size: bool = False,
                    target_height: int = 64) -> np.ndarray:
    """Run full normalization pipeline on an in-memory BGR image; returns grayscale."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if len(img_bgr.shape) == 3 else img_bgr.copy()
    if binarize:
        gray = binarize_image(gray, method="adaptive")
    if denoise:
        gray = remove_noise(gray)
    if crop:
        gray = crop_to_content(gray)
    if normalize_size:
        gray = normalize_height(gray, target_height=target_height)
    return gray

## Run Full Pipeline

In [21]:
pdf_files = sorted(glob.glob(os.path.join(INPUT_PDF_DIR, "*.pdf")))
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {INPUT_PDF_DIR!r}")

for pdf_path in pdf_files:
    base = os.path.splitext(os.path.basename(pdf_path))[0]
    print(f"Processing: {pdf_path}")

    lined_page, blank_page = select_pages(pdf_path)

    # ── Lined page: crop -> Final Data, normalize -> FinalDataNormalized ──
    print(f"  Lined page: {lined_page.shape[1]}x{lined_page.shape[0]} px")
    line_ys  = detect_printed_lines(lined_page, debug=False)
    x_bounds = detect_content_x_bounds(lined_page)
    print(f"  Found {len(line_ys)} printed lines -> {len(line_ys)-1} crops")

    if len(line_ys) < 2:
        print("  Not enough lines detected — skipping lined page")
    else:
        crop_paths = crop_lines(lined_page, line_ys, FINAL_DATA_DIR, base, x_bounds=x_bounds)
        print(f"  Saved {len(crop_paths)} line crops to {FINAL_DATA_DIR!r}")

        for crop_path in crop_paths:
            crop_img  = cv2.imread(crop_path)
            normalized = normalize_image(crop_img)
            out_path  = os.path.join(FINAL_DATA_NORM_DIR, os.path.basename(crop_path))
            cv2.imwrite(out_path, normalized)

        print(f"  Normalized {len(crop_paths)} crops -> {FINAL_DATA_NORM_DIR!r}")

    # ── Blank page: normalize then column detection -> FinalDataNormalized ──
    if blank_page is not None:
        print(f"  Blank page: {blank_page.shape[1]}x{blank_page.shape[0]} px")
        norm_gray = normalize_image(blank_page, normalize_size=False)
        norm_bgr  = cv2.cvtColor(norm_gray, cv2.COLOR_GRAY2BGR)
        column_detection(norm_bgr, FINAL_DATA_NORM_DIR, base)

Processing: whole image\sd.pdf
  Line scores: [94863, 48194]  ->  lined=page 1, blank=page 2
  Lined page: 2484x3509 px
  Frame bounds detected: x_left=23, x_right=2452
  Found 22 printed lines -> 21 crops
  Saved 21 line crops to 'Final Data'
  Normalized 21 crops -> 'FinalDataNormalized'
  Blank page: 2484x3509 px
  Detected 3 column(s) -> saved sd_COLUMNS.png
